In [3]:
#!pip install statsmodels

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 13.9 MB/s  0:00:00.6 MB/s eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [statsmodels] 1/2 [statsmodels]


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
import warnings
warnings.filterwarnings('ignore')

# Read the data
try:
    df = pd.read_csv('bangkok-air-quality.csv')

    # Clean column names
    df.columns = df.columns.str.strip()

    # Ensure date is datetime
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date').set_index('date')

    # Replace empty spaces or non-numeric with NaN in 'pm25'
    df['pm25'] = pd.to_numeric(df['pm25'], errors='coerce')

    # Filter data from 2016 onwards as per the project scope
    df = df[df.index.year >= 2016]

    # Handle Missing Values (Mean Imputation)
    mean_pm25 = df['pm25'].mean()
    df['pm25'] = df['pm25'].fillna(mean_pm25)

    # Remove Outliers (IQR method)
    Q1 = df['pm25'].quantile(0.25)
    Q3 = df['pm25'].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df_clean = df[(df['pm25'] >= lower) & (df['pm25'] <= upper)].copy()

    # Thai Seasons Function
    def get_season(month):
        if month in [3, 4, 5]: return 'Summer'
        elif month in [6, 7, 8, 9, 10]: return 'Rainy'
        else: return 'Winter'

    df_clean['month'] = df_clean.index.month
    df_clean['season'] = df_clean['month'].apply(get_season)

    # Statistical Modeling (Descriptive Stats)
    stats = df_clean.groupby('season')['pm25'].agg(['mean', 'max', 'std']).round(2)
    print("--- Seasonal Stats ---")
    print(stats)

    # Month analysis for Key Findings
    monthly_stats = df_clean.groupby('month')['pm25'].mean().round(2)
    worst_month = monthly_stats.idxmax()
    print(f"\nWorst month index: {worst_month}")

    # EDA: Time-Series Decomposition
    df_ts = df_clean['pm25'].resample('D').mean().interpolate(method='time')

    # Plot 1: Decomposition
    decomposition = seasonal_decompose(df_ts.dropna(), model='additive', period=365)
    fig = decomposition.plot()
    fig.set_size_inches(12, 8)
    plt.suptitle('PM2.5 Time-Series Decomposition (Bangkok 2016-Present)', fontsize=14)
    plt.tight_layout()
    plt.savefig('pm25_decomposition.png')
    plt.show()

    # Plot 2: Seasonal Boxplot
    plt.figure(figsize=(10, 6))
    sns.boxplot(x='season', y='pm25', data=df_clean, order=['Winter', 'Summer', 'Rainy'], palette='Set2')
    plt.title('PM2.5 Distribution by Thai Seasons (2016-Present)')
    plt.ylabel('PM2.5 Concentration (AQI)')
    plt.xlabel('Season')
    plt.savefig('pm25_seasonal_boxplot.png')
    plt.show()

    # Plot 3: 10-year trend (Monthly Average)
    plt.figure(figsize=(14, 6))
    df_clean['pm25'].resample('ME').mean().plot(color='firebrick', linewidth=2)
    plt.title('Monthly Average PM2.5 in Bangkok (2016-Present)')
    plt.ylabel('Average PM2.5 (AQI)')
    plt.xlabel('Year')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.savefig('pm25_10yr_trend.png')
    plt.show()

    print("\nPlots generated successfully.")
except Exception as e:
    print(f"Error: {e}")

In [ ]:
from scipy.stats import ttest_ind
import itertools
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Season order for the plot
season_order = ['Winter', 'Summer', 'Rainy']
alpha = 0.05

# --- Compute pairwise T-tests with inference ---
pairs = list(itertools.combinations(season_order, 2))
results = {}

print("=" * 70)
print("STATISTICAL INFERENCE TEST: Pairwise Welch's T-Test")
print(f"Significance Level (alpha): {alpha}")
print("=" * 70)

for s1, s2 in pairs:
    g1 = df_clean[df_clean['season'] == s1]['pm25'].dropna()
    g2 = df_clean[df_clean['season'] == s2]['pm25'].dropna()

    # Welch's t-test (does not assume equal variances)
    stat, p = ttest_ind(g1, g2, equal_var=False)

    # Compute 95% confidence interval for the difference in means
    mean_diff = g1.mean() - g2.mean()
    se_diff = np.sqrt((g1.std()**2 / len(g1)) + (g2.std()**2 / len(g2)))
    from scipy.stats import t as t_dist
    # Welch-Satterthwaite degrees of freedom
    df_num = ((g1.std()**2 / len(g1)) + (g2.std()**2 / len(g2)))**2
    df_den = ((g1.std()**2 / len(g1))**2 / (len(g1) - 1)) + ((g2.std()**2 / len(g2))**2 / (len(g2) - 1))
    dof = df_num / df_den
    t_crit = t_dist.ppf(1 - alpha / 2, dof)
    ci_lower = mean_diff - t_crit * se_diff
    ci_upper = mean_diff + t_crit * se_diff

    results[(s1, s2)] = (stat, p, mean_diff, ci_lower, ci_upper, dof)

    # Print inference
    print(f"\n--- {s1} vs {s2} ---")
    print(f"  H0: Mean PM2.5 of {s1} = Mean PM2.5 of {s2} (no difference)")
    print(f"  H1: Mean PM2.5 of {s1} \u2260 Mean PM2.5 of {s2} (there is a difference)")
    print(f"  {s1} mean: {g1.mean():.2f}  |  {s2} mean: {g2.mean():.2f}")
    print(f"  Mean Difference: {mean_diff:.2f}")
    print(f"  Degrees of Freedom: {dof:.1f}")
    print(f"  T-statistic: {stat:.4f}")
    if p < 0.001:
        print(f"  P-value: < 0.001")
    else:
        print(f"  P-value: {p:.4f}")
    print(f"  95% CI for difference: [{ci_lower:.2f}, {ci_upper:.2f}]")
    if p < alpha:
        print(f"  >> REJECT H0: The difference is statistically significant at alpha = {alpha}.")
    else:
        print(f"  >> FAIL TO REJECT H0: No significant difference at alpha = {alpha}.")

print("\n" + "=" * 70)

# --- Create box plot with T-test annotations ---
fig, ax = plt.subplots(figsize=(10, 8))
sns.boxplot(x='season', y='pm25', data=df_clean, order=season_order, palette='Set2', ax=ax)

ax.set_title('PM2.5 Distribution by Thai Seasons\nwith Welch\'s T-Test Inference Results', fontsize=14, fontweight='bold')
ax.set_ylabel('PM2.5 Concentration (AQI)')
ax.set_xlabel('Season')

# --- Draw significance brackets ---
y_max = df_clean['pm25'].max()
h = 4
gap = 12

for i, (s1, s2) in enumerate(pairs):
    x1 = season_order.index(s1)
    x2 = season_order.index(s2)
    stat, p, mean_diff, ci_lower, ci_upper, dof = results[(s1, s2)]

    # Format label with t-stat, p-value, and decision
    if p < 0.001:
        p_str = 'p < 0.001'
    else:
        p_str = f'p = {p:.4f}'

    decision = "Reject H0" if p < alpha else "Fail to Reject H0"
    label = f"t = {stat:.2f}, {p_str} \u2192 {decision}"

    y = y_max + gap + i * (h + gap + 6)

    # Draw bracket lines
    ax.plot([x1, x1, x2, x2], [y, y + h, y + h, y], lw=1.5, color='black')
    ax.text((x1 + x2) / 2, y + h + 1, label, ha='center', va='bottom', fontsize=9, fontweight='bold',
            color='red' if p < alpha else 'gray')

# Adjust y-axis to fit brackets
ax.set_ylim(top=y_max + gap + len(pairs) * (h + gap + 6) + 15)

plt.tight_layout()
plt.savefig('pm25_boxplot_ttest.png', dpi=150)
plt.show()
